In [0]:
from pyspark.sql import functions as F
import time

spark.conf.set("spark.sql.session.timeZone", "UTC")
SILVER_VP = "transit.silver.vehicle_positions"
FACT      = "transit.gold.fact_vehicle_position"

def dim_lookup(table, natural, key):
    """Natural key -> surrogate key, real members only. The unknown member is the fallback."""
    return spark.table(table).filter(F.col(key) != -1).select(natural, key)

dims = {
    "route": dim_lookup("transit.gold.dim_route", "route_id", "route_key"),
    "trip":  dim_lookup("transit.gold.dim_trip",  "trip_id",  "trip_key"),
    "stop":  dim_lookup("transit.gold.dim_stop",  "stop_id",  "stop_key"),
}
dates = (spark.table("transit.gold.dim_date").filter("date_key != -1")
           .select("date_key", F.lit(True).alias("_date_found")))

for name, d in dims.items():
    nat = d.columns[0]
    n, distinct = d.count(), d.select(nat).distinct().count()
    print(f"dim_{name:6} {n:>8,} members · natural keys unique: {n == distinct}")
    assert n == distinct, f"dim_{name}: duplicate natural keys would multiply fact rows"

In [0]:
vp = spark.table(SILVER_VP)

fact = (vp
    .join(F.broadcast(dims["route"]), "route_id", "left")
    .join(F.broadcast(dims["trip"]),  "trip_id",  "left")
    .join(F.broadcast(dims["stop"]),  "stop_id",  "left")
    .withColumn("date_key", F.date_format("obs_date_local", "yyyyMMdd").cast("int"))
    .join(F.broadcast(dates), "date_key", "left")
    .select(
        # foreign keys: -1 when unresolved
        F.when(F.col("_date_found"), F.col("date_key")).otherwise(F.lit(-1)).alias("date_key"),
        F.coalesce("route_key", F.lit(-1)).alias("route_key"),
        F.when(F.col("trip_key").isNotNull(), F.col("trip_key"))
         .when(F.col("schedule_relationship") == "ADDED", F.lit(-2))
         .otherwise(F.lit(-1)).alias("trip_key"),
        F.coalesce("stop_key",  F.lit(-1)).alias("stop_key"),
        # natural ids and descriptive attributes kept on the fact, for tracing
        "vehicle_id", "vehicle_label", "route_id", "trip_id", "stop_id",
        "vehicle_ts", "vehicle_ts_utc",
        "current_status", "occupancy_status", "schedule_relationship",
        "is_revenue", "direction_id", "stop_sequence",
        # measures
        "latitude", "longitude", "bearing", "speed", "occupancy_pct",
        (F.col("first_seen_snapshot_ts") - F.col("vehicle_ts")).alias("report_age_s"),
        (F.col("last_seen_snapshot_ts") - F.col("first_seen_snapshot_ts")).alias("frozen_s"),
        "first_seen_snapshot_ts", "last_seen_snapshot_ts"))

print(f"{len(fact.columns)} columns")

In [0]:
n_silver, n_fact = vp.count(), fact.count()
print(f"silver {n_silver:,} · fact {n_fact:,}")
assert n_fact == n_silver, "a join multiplied or dropped rows"

t0 = time.time()
(fact.withColumn("_gold_loaded_at", F.current_timestamp())
   .write.format("delta").mode("overwrite")
   .option("overwriteSchema", "true")
   .partitionBy("date_key")
   .saveAsTable(FACT))

d = spark.sql(f"DESCRIBE DETAIL {FACT}").select("numFiles", "sizeInBytes").first()
print(f"{FACT}: {time.time()-t0:.0f}s · {d.numFiles} files · {d.sizeInBytes/1e6:.1f} MB")

In [0]:
f = spark.table(FACT)

for key, dim in [("date_key",  "transit.gold.dim_date"),
                 ("route_key", "transit.gold.dim_route"),
                 ("trip_key",  "transit.gold.dim_trip"),
                 ("stop_key",  "transit.gold.dim_stop")]:
    orphans = (f.select(key).distinct()
                .join(spark.table(dim).select(key), key, "left_anti").count())
    print(f"{key:10} orphan keys: {orphans}")
    assert orphans == 0, f"{key} values with no matching dimension row"

print("\nevery foreign key resolves to a dimension row (real or unknown member)")

In [0]:
total = f.count()
rows = []
for key, nat in [("route_key", "route_id"), ("trip_key", "trip_id"), ("stop_key", "stop_id")]:
    unknown = f.filter(F.col(key) == -1)
    blank   = unknown.filter(F.col(nat).isNull()).count()
    missing = unknown.filter(F.col(nat).isNotNull()).count()
    rows.append((key, blank, missing, round(100 * (blank + missing) / total, 2)))
rows.append(("date_key", 0, f.filter("date_key = -1").count(),
             round(100 * f.filter("date_key = -1").count() / total, 2)))

spark.createDataFrame(rows, "fk string, blank_at_source long, not_in_dimension long, "
                            "pct_unknown double").display()

In [0]:
unres = f.filter("trip_key = -1 AND trip_id IS NOT NULL")

# 1. By date: is the problem concentrated in older days?
(f.groupBy("date_key")
   .agg(F.count("*").alias("observations"),
        F.sum((F.col("trip_key") == -1).cast("int")).alias("unknown_trip"))
   .withColumn("pct_unknown", F.round(100 * F.col("unknown_trip") / F.col("observations"), 1))
   .orderBy("date_key").display())

# 2. By cause: were the missing trips ever in any published schedule?
versions = spark.table("transit.bronze.gtfs_trips").select("trip_id", "_feed_version").distinct()
cause = (unres.select("trip_id").distinct()
   .join(versions, "trip_id", "left")
   .groupBy("trip_id")
   .agg(F.sort_array(F.collect_set("_feed_version")).alias("in_versions"))
   .withColumn("cause", F.when(F.size("in_versions") == 0, "never in any published schedule")
                         .otherwise(F.concat(F.lit("only in "), F.array_join("in_versions", ", ")))))

(unres.join(cause, "trip_id")
   .groupBy("cause")
   .agg(F.countDistinct("trip_id").alias("distinct_trips"), F.count("*").alias("fact_rows"))
   .orderBy(F.desc("fact_rows")).display())

# 3. The never-published ones: which routes, and are they the shuttles?
(unres.join(cause.filter("size(in_versions) = 0"), "trip_id")
   .groupBy("route_id", "schedule_relationship").count()
   .orderBy(F.desc("count")).limit(15).display())

In [0]:
spark.sql("""
  SELECT r.route_type_name,
         d.day_name,
         d.is_weekend,
         count(*)                        AS observations,
         round(avg(f.occupancy_pct), 1)  AS avg_occupancy_pct
  FROM transit.gold.fact_vehicle_position f
  JOIN transit.gold.dim_route r ON f.route_key = r.route_key
  JOIN transit.gold.dim_date  d ON f.date_key  = d.date_key
  WHERE f.is_revenue AND f.occupancy_pct IS NOT NULL
  GROUP BY r.route_type_name, d.day_name, d.is_weekend, d.day_of_week
  ORDER BY r.route_type_name, d.day_of_week
""").display()